# Qwen3.8-27B teacher distillation on Google Colab

This notebook generates replacement Python training data for the `Qwen/Qwen3.5-4B`
student using `Qwen/Qwen3.8-27B` as the teacher on a Google Colab A100.

**Colab is only a disposable GPU worker.** It does not need GitHub credentials,
SSH keys, `git clone`, `git pull`, or `git push`.

Before starting, create or download a ZIP of the exact `tiny-qwen-coder` repository
version you want to run. This notebook stores and verifies that frozen ZIP on Google
Drive so the same code can be reconstructed after a Colab preemption.

Recommended progression:

1. 16 records — runtime and checkpoint smoke test.
2. 500 records — output/finalization inspection.
3. 2,000 records — first real learning experiment.
4. Scale further only if the 2,000-record experiment beats the frozen base benchmark.

## 1. Mount Google Drive

Select an **A100 GPU** runtime in Colab first, then run this cell.

In [ ]:
from google.colab import drive

drive.mount("/content/drive")

## 2. Define durable Drive paths

The repository ZIP, input corpus, subsets, checkpoints, and finalized data all live
on Google Drive. `/content` is treated as disposable scratch space.

In [ ]:
import os
from pathlib import Path

os.environ["TQC_DRIVE"] = "/content/drive/MyDrive/tiny-qwen-coder"
os.environ["CODE_DIR"] = f"{os.environ['TQC_DRIVE']}/code"
os.environ["CODE_ARCHIVE"] = f"{os.environ['CODE_DIR']}/tiny-qwen-coder.zip"
os.environ["RUN_ROOT"] = f"{os.environ['TQC_DRIVE']}/distillation/qwen38-27b-v1"
os.environ["INPUT_DIR"] = f"{os.environ['RUN_ROOT']}/input"
os.environ["SUBSET_DIR"] = f"{os.environ['RUN_ROOT']}/subsets"
os.environ["SMOKE_DIR"] = f"{os.environ['TQC_DRIVE']}/distillation/qwen38-27b-v1-smoke"
os.environ["PILOT_DIR"] = f"{os.environ['TQC_DRIVE']}/distillation/qwen38-27b-v1-2000"

for key in (
    "CODE_DIR",
    "RUN_ROOT",
    "INPUT_DIR",
    "SUBSET_DIR",
    "SMOKE_DIR",
    "PILOT_DIR",
):
    Path(os.environ[key]).mkdir(parents=True, exist_ok=True)

print("Repository ZIP expected at:", os.environ["CODE_ARCHIVE"])

## 3. Put the frozen repository ZIP on Drive

Upload your repository archive to:

`MyDrive/tiny-qwen-coder/code/tiny-qwen-coder.zip`

If the file is not already there, this optional cell lets you select the ZIP from
your computer and copies it to the durable Drive location.

Once a real generation run has started, **do not replace this ZIP with newer code**.
Use a new archive and a new checkpoint directory for a different code version.

In [ ]:
from pathlib import Path

archive = Path(os.environ["CODE_ARCHIVE"])

if archive.exists():
    print("Repository ZIP already exists:", archive)
else:
    from google.colab import files

    uploaded = files.upload()
    if len(uploaded) != 1:
        raise RuntimeError("Upload exactly one repository ZIP.")

    uploaded_name, uploaded_bytes = next(iter(uploaded.items()))
    if not uploaded_name.lower().endswith(".zip"):
        raise RuntimeError("The uploaded file must be a .zip archive.")

    archive.write_bytes(uploaded_bytes)
    print("Saved repository ZIP to:", archive)

## 4. Verify and extract the frozen repository ZIP

On first use, this writes `tiny-qwen-coder.zip.sha256` next to the archive on Drive.
On every later Colab allocation, the ZIP must match that checksum exactly.

The cell also finds the repository root automatically whether the archive contains
the files directly or wraps them in a directory such as `tiny-qwen-coder-master/`.

In [ ]:
import hashlib
import shutil
from pathlib import Path
from zipfile import ZipFile

archive = Path(os.environ["CODE_ARCHIVE"])
if not archive.is_file():
    raise FileNotFoundError(
        f"Repository ZIP not found: {archive}\n"
        "Upload tiny-qwen-coder.zip to the Google Drive code directory first."
    )


def sha256_file(path: Path) -> str:
    digest = hashlib.sha256()
    with path.open("rb") as handle:
        for chunk in iter(lambda: handle.read(1024 * 1024), b""):
            digest.update(chunk)
    return digest.hexdigest()


archive_sha256 = sha256_file(archive)
checksum_path = archive.with_suffix(archive.suffix + ".sha256")

if checksum_path.exists():
    expected_sha256 = checksum_path.read_text(encoding="ascii").split()[0]
    if archive_sha256 != expected_sha256:
        raise RuntimeError(
            "Repository ZIP checksum changed. Do not resume this experiment with different code."
        )
else:
    checksum_path.write_text(
        f"{archive_sha256}  {archive.name}\n",
        encoding="ascii",
    )

scratch_root = Path("/content/tiny-qwen-coder-code")
if scratch_root.exists():
    shutil.rmtree(scratch_root)
scratch_root.mkdir(parents=True)

with ZipFile(archive) as zip_file:
    zip_file.extractall(scratch_root)

repo_candidates = sorted(
    {
        pyproject.parent
        for pyproject in scratch_root.rglob("pyproject.toml")
        if (pyproject.parent / "scripts/teacher_distillation/README.md").is_file()
    }
)
if len(repo_candidates) != 1:
    raise RuntimeError(
        "Expected exactly one tiny-qwen-coder repository in the ZIP; "
        f"found {len(repo_candidates)} candidates."
    )

repo = repo_candidates[0]
os.environ["TQC_REPO"] = str(repo)
os.environ["TQC_CODE_ARCHIVE_SHA256"] = archive_sha256

print("repository:", repo)
print("archive SHA-256:", archive_sha256)
print("checksum:", checksum_path)

## 5. Install the repository and the pinned teacher runtime

This installs the extracted project plus the Colab-only Qwen3.8/vLLM dependencies.

In [ ]:
import os

os.chdir(os.environ["TQC_REPO"])
print("working directory:", os.getcwd())

In [ ]:
!python -m pip install -e .
!python -m pip install -r requirements/colab-teacher.txt

## 6. Verify the GPU

Stop if this does not report the expected CUDA GPU/A100.

In [ ]:
!nvidia-smi

import torch

print("torch:", torch.__version__)
print("cuda:", torch.version.cuda)
print("gpu:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "NONE")

if not torch.cuda.is_available():
    raise RuntimeError("CUDA GPU is not available.")

## 7. Build and seal the immutable teacher input

The original P0 assistant responses are not sent to the teacher. This builds the
canonical prompt-only input and writes it to Drive with a SHA-256 sidecar.

Run this once for an experiment. Do not rebuild or edit it after generation starts.

In [ ]:
!python scripts/teacher_distillation/prepare_teacher_input.py \
  --output "$INPUT_DIR/accepted.jsonl"

## 8. Create deterministic pilot subsets

These are source-stratified deterministic samples; do not use the first N source
rows as a scientific pilot.

In [ ]:
!python scripts/teacher_distillation/select_teacher_input.py \
  --input "$INPUT_DIR/accepted.jsonl" \
  --output "$SUBSET_DIR/p0-16.jsonl" \
  --count 16

!python scripts/teacher_distillation/select_teacher_input.py \
  --input "$INPUT_DIR/accepted.jsonl" \
  --output "$SUBSET_DIR/p0-500.jsonl" \
  --count 500

!python scripts/teacher_distillation/select_teacher_input.py \
  --input "$INPUT_DIR/accepted.jsonl" \
  --output "$SUBSET_DIR/p0-2000.jsonl" \
  --count 2000

## 9. Run the 16-record A100 smoke test

**Run this before any larger generation.** A successful smoke should finish all
16 records and leave one sealed shard on Google Drive.

In [ ]:
!python scripts/teacher_distillation/generate_teacher_data.py \
  --input "$SUBSET_DIR/p0-16.jsonl" \
  --checkpoint-dir "$SMOKE_DIR/checkpoint" \
  --work-dir /content/tqc-distillation-smoke

### Check smoke-test status without loading Qwen3.8 again

In [ ]:
!python scripts/teacher_distillation/generate_teacher_data.py \
  --input "$SUBSET_DIR/p0-16.jsonl" \
  --checkpoint-dir "$SMOKE_DIR/checkpoint" \
  --work-dir /content/tqc-distillation-smoke \
  --status-only

## 10. Optional 500-record inspection run

Use this after the 16-record smoke succeeds if you want an intermediate corpus for
manual output inspection and finalization-loss analysis.

This uses its own checkpoint directory so it cannot contaminate the 2,000-record run.

In [ ]:
os.environ["INSPECT_DIR"] = f"{os.environ['TQC_DRIVE']}/distillation/qwen38-27b-v1-500"
Path(os.environ["INSPECT_DIR"]).mkdir(parents=True, exist_ok=True)

!python scripts/teacher_distillation/generate_teacher_data.py \
  --input "$SUBSET_DIR/p0-500.jsonl" \
  --checkpoint-dir "$INSPECT_DIR/checkpoint" \
  --work-dir /content/tqc-distillation-500

## 11. Generate the 2,000-record pilot

**Do not run this until the 16-record smoke has succeeded.**

This is the first real learning experiment. If Colab is preempted, remount the same
Drive, rerun setup through GPU verification using the same frozen ZIP, then rerun
this exact cell. Completed sealed shards are verified and skipped automatically.

In [ ]:
!python scripts/teacher_distillation/generate_teacher_data.py \
  --input "$SUBSET_DIR/p0-2000.jsonl" \
  --checkpoint-dir "$PILOT_DIR/checkpoint" \
  --work-dir /content/tqc-distillation-2000

### Check 2,000-record progress without loading Qwen3.8

In [ ]:
!python scripts/teacher_distillation/generate_teacher_data.py \
  --input "$SUBSET_DIR/p0-2000.jsonl" \
  --checkpoint-dir "$PILOT_DIR/checkpoint" \
  --work-dir /content/tqc-distillation-2000 \
  --status-only

## 12. Finalize the 2,000-record pilot

Run this only after generation reports `2000/2000`.

The resulting Drive directory contains the accepted corpus, deterministic
train/validation splits, checksums, dataset manifest, and finalization report.
That directory is what you bring back to the RTX 4070 Ti training machine.

In [ ]:
!python scripts/teacher_distillation/finalize_teacher_data.py \
  --input "$SUBSET_DIR/p0-2000.jsonl" \
  --checkpoint-dir "$PILOT_DIR/checkpoint" \
  --output-dir "$PILOT_DIR/final"

## Recovery rules

- Treat `/content` as disposable and Google Drive as durable.
- Keep the repository ZIP and its `.sha256` sidecar on Drive.
- Reuse the exact same ZIP and sealed input when resuming.
- Keep 16/500/2,000/full experiments in separate checkpoint directories.
- Do not overwrite the repository ZIP after a real generation run starts.
- Never edit `run-identity.json`, generated shards, or checksum sidecars to force progress.
- An uncommitted shard left by a killed runtime is regenerated automatically.
- A sealed shard with a bad checksum fails closed as corruption.
- Do not generate the full ~40k corpus until the 2,000-record experiment shows an improvement over the frozen base benchmark.

There is intentionally no GitHub write workflow in Colab. Development, commits,
pushes, and pulls belong on the normal development machine.